In [ ]:
@app.get("/ask")
async def ask(q: str):
    cached = await redis_client.get(q)
    if cached:
        return {"source": "cache", "answer": cached}

    api_task = asyncio.create_task(fetch_external_data(q))
    llm_task = asyncio.create_task(chain.ainvoke({"question": q}))

    api_result, llm_result = await asyncio.gather(api_task, llm_task)

    answer = {
        "api": api_result,
        "llm": llm_result,
    }

    await redis_client.set(q, json.dumps(answer), ex=3600)

    return {"source": "fresh", "answer": answer}

In [1]:
import asyncio
import httpx

In [2]:
async def fetch(client, url):
    response = client.get(url)
    return response.status_code

In [3]:
from fastapi import FastAPI
import httpx

app = FastAPI()

@app.get("/external")
async def external():
    urls = [
        "https://www.example.com/",
        "https://www.python.org/",
        "https://www.httpx.org/",
    ]

    async with httpx.AsyncClient() as client:
        tasks = [fetch(client, url) for url in urls]
        results = await asyncio.gather(*tasks)


    print(results)

In [ ]:
import httpx
import asyncio

async def main():
    async with httpx.AsyncClient() as client:
        async with client.stream("GET", "https://www.example.com/") as response:
            async for chunk in response.aiter_bytes():
                print(chunk)

asyncio.run(main())


In [ ]:
import httpx
import asyncio

async def huge_file_download():
    async with httpx.AsyncClient() as client:
        async with client.stream("GET", "https://www.example.com/") as response:
            async for chunk in response.aiter_bytes():
                print(chunk)

asyncio.run(main())

In [ ]:
async with r.pipeline(transaction=True) as pipe:
    pipe.set("a", "1")
    pipe.set("a")
    await pipe.execute()

只问一次，等完整结果：
await chain.ainvoke(input)

多个输入一起跑：
await chain.abatch([input1, input2, input3])

想边生成边显示：
async for chunk in chain.astream(input):

    ...

In [ ]:
async def main():
    inputs = [
        {"topic": "asyncio"},
        {"topic": "FastAPI"},
        {"topic": "Redis"},
    ]

    results = await chain.abatch(inputs)

    for result in results:
        print(result)

In [ ]:
async def main():
    async for chunk in chain.astream({"topic": "asyncio"}):
        print(chunk, end="", flush=True)

asyncio.run(main())

In [ ]:
from fastapi import FastAPI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

app = FastAPI()

prompt = ChatPromptTemplate.from_template("Answer this question: {question}")
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

chain = prompt | model | parser

@app.post("/ask")
async def ask(question: str):
    answer = await chain.ainvoke({"question": question})
    return {"answer": answer}

In [ ]:
from pydantic import BaseModel

class BatchRequest(BaseModel):
    questions: list[str]

@app.post("/ask-batch")
async def ask_batch(req: BatchRequest):
    inputs = [{"question": q} for q in req.questions]

    answers = await chain.abatch(
        inputs,
        config={"max_concurrency": 5},
    )

    return {"answers": answers}

In [ ]:
from fastapi.responses import StreamingResponse

@app.get("/stream")
async def stream(question: str):
    async def generate():
        async for chunk in chain.astream({"question": question}):
            yield chunk

    return StreamingResponse(generate(), media_type="text/plain")